# ProNet – KNN Professional Connection Recommendation
**Machine Learning Module** | Algorithm: K-Nearest Neighbors


## Step 1: Install & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import joblib, os
print('All libraries imported!')

## Step 2: Create Dataset (200 Professional Profiles)

In [ ]:
np.random.seed(42)
industries = ['Technology','Finance','Healthcare','Marketing','Design','Data Science']
roles = ['Software Engineer','Data Scientist','Product Manager','Designer','Business Analyst','ML Engineer']
skills = ['Python','Java','React','SQL','Machine Learning','Figma','Excel','AWS','Docker','Tableau']

rows = []
for i in range(200):
    ind = np.random.choice(industries)
    exp = np.random.randint(0, 20)
    rows.append({
        'industry': ind,
        'role': np.random.choice(roles),
        'skill': np.random.choice(skills),
        'exp_years': exp,
        'exp_level': 'Entry' if exp<3 else ('Mid' if exp<8 else 'Senior')
    })

df = pd.DataFrame(rows)
os.makedirs('dataset', exist_ok=True)
df.to_csv('dataset/profiles.csv', index=False)
print(df.shape)
df.head(10)

## Step 3: Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(12,4))
df['industry'].value_counts().plot(kind='bar', ax=ax[0], color='steelblue', title='Profiles by Industry')
df['exp_level'].value_counts().plot(kind='pie', ax=ax[1], autopct='%1.1f%%', title='Experience Levels')
plt.tight_layout(); plt.show()

## Step 4: Encode Features (Text → Numbers)

In [ ]:
le_ind = LabelEncoder(); le_role = LabelEncoder()
le_skill = LabelEncoder(); le_exp = LabelEncoder()

df['industry_enc']  = le_ind.fit_transform(df['industry'])
df['role_enc']      = le_role.fit_transform(df['role'])
df['skill_enc']     = le_skill.fit_transform(df['skill'])
df['exp_level_enc'] = le_exp.fit_transform(df['exp_level'])

X = df[['industry_enc','role_enc','skill_enc','exp_level_enc','exp_years']].values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
print('Feature matrix:', X_scaled.shape)
pd.DataFrame(X_scaled, columns=['industry','role','skill','exp_level','exp_years']).head()

## Step 5: Train KNN Model

In [ ]:
knn = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn.fit(X_scaled)
print('KNN model trained! k=5, cosine similarity metric')

## Step 6: Evaluate Model

In [ ]:
correct=0; total=100
for _ in range(total):
    idx = np.random.randint(0,200)
    d,inds = knn.kneighbors(X_scaled[idx].reshape(1,-1), n_neighbors=6)
    nbrs = inds[0][1:]
    same = sum(1 for i in nbrs if df.iloc[i]['industry']==df.iloc[idx]['industry'])
    if same>=2: correct+=1
print(f'Same-industry Precision@5: {correct/total:.0%}')

## Step 7: Demo - Get Recommendations for a Test User

In [ ]:
test_vec = [
    le_ind.transform(['Technology'])[0],
    le_role.transform(['Software Engineer'])[0],
    le_skill.transform(['Python'])[0],
    le_exp.transform(['Mid'])[0],
    4
]
ts = scaler.transform([test_vec])
d, inds = knn.kneighbors(ts, n_neighbors=6)
print('Top 5 Recommended Connections:')
for i,idx in enumerate(inds[0][1:]):
    p=df.iloc[idx]; score=round((1-d[0][i+1])*100,1)
    print(f'{i+1}. {p["role"]} | {p["industry"]} | {p["exp_years"]}yrs | Match: {score}%')

## Step 8: Save Model with joblib

In [ ]:
os.makedirs('model', exist_ok=True)
joblib.dump({'knn':knn,'scaler':scaler,'le_ind':le_ind,'le_role':le_role,'le_skill':le_skill,'le_exp':le_exp},'model/knn_model.pkl')
print('Model saved to model/knn_model.pkl')